Serialization

In [1]:
from pydantic import BaseModel, Field, EmailStr, AnyUrl, field_validator, model_validator, computed_field
from typing import List, Dict, Optional, Annotated

In [2]:
class Address(BaseModel):
    city: str
    state: str
    pin: str

In [3]:
address_info = {'city': 'New York', 'state': 'NY', 'pin': '10001'}
address1 = Address(**address_info)
print(address1)

city='New York' state='NY' pin='10001'


In [4]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    height: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]
    address: Optional[Address]

    @field_validator('email')
    @classmethod
    def validate_email(cls, value):
        allowed_domains = ['sbi.com', 'equitas.com']
        domain = value.split('@')[-1]
        if domain not in allowed_domains:
            raise ValueError(f"Email domain must be one of {allowed_domains}")
        return value
    
    @field_validator('name', mode='after')
    @classmethod
    def transform_name(cls, value):
        return value.strip().upper()
    
    @field_validator('age')
    @classmethod
    def validate_age(cls, value):
        if value < 18:
            raise ValueError("Patient must be at least 18 years old")
        return value
    
    @model_validator(mode='after')
    def validate_emergency_contact(cls, model):
        if model.age > 60 and 'emergency_contact' not in model.contact_info:
            raise ValueError("Patients over 60 must have an emergency contact")
        return model
    
    @computed_field
    @property
    def bmi(self) -> float:
        return round(self.weight / ((self.height / 100) ** 2), 2)

In [5]:
patient_info = {'name': 'benky', 'email': 'benky@sbi.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '65', 'weight': 70.5, 'height': 175.0, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890', 'emergency_contact': '9876543210'}, 'address': address1}
patient1 = Patient(**patient_info)
print(patient1)

name='BENKY' email='benky@sbi.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=65 weight=70.5 height=175.0 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890', 'emergency_contact': '9876543210'} address=Address(city='New York', state='NY', pin='10001') bmi=23.02


In [10]:
temp = patient1.model_dump()
print(temp)
print(type(temp))

{'name': 'BENKY', 'email': 'benky@sbi.com', 'linkedin_url': AnyUrl('https://linkedin.com/in/benky'), 'age': 65, 'weight': 70.5, 'height': 175.0, 'married': False, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890', 'emergency_contact': '9876543210'}, 'address': {'city': 'New York', 'state': 'NY', 'pin': '10001'}, 'bmi': 23.02}
<class 'dict'>


In [9]:
temp = patient1.model_dump_json()
print(temp)
print(type(temp))

{"name":"BENKY","email":"benky@sbi.com","linkedin_url":"https://linkedin.com/in/benky","age":65,"weight":70.5,"height":175.0,"married":false,"allergies":["dust","pollen"],"contact_info":{"phone":"1234567890","emergency_contact":"9876543210"},"address":{"city":"New York","state":"NY","pin":"10001"},"bmi":23.02}
<class 'str'>


In [11]:
temp = patient1.model_dump(include=["name"])
print(temp)

{'name': 'BENKY'}


In [12]:
temp = patient1.model_dump(include=["name", "age"])
print(temp)

{'name': 'BENKY', 'age': 65}


In [13]:
temp = patient1.model_dump(exclude=["name", "age"])
print(temp)

{'email': 'benky@sbi.com', 'linkedin_url': AnyUrl('https://linkedin.com/in/benky'), 'weight': 70.5, 'height': 175.0, 'married': False, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890', 'emergency_contact': '9876543210'}, 'address': {'city': 'New York', 'state': 'NY', 'pin': '10001'}, 'bmi': 23.02}


In [14]:
temp = patient1.model_dump(exclude={"address": "state"})
print(temp)

{'name': 'BENKY', 'email': 'benky@sbi.com', 'linkedin_url': AnyUrl('https://linkedin.com/in/benky'), 'age': 65, 'weight': 70.5, 'height': 175.0, 'married': False, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890', 'emergency_contact': '9876543210'}, 'address': {'city': 'New York', 'pin': '10001'}, 'bmi': 23.02}


Can restrict fields that are not defined while instantiating

In [15]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    height: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]
    address: Optional[Address]
    gender: Optional[str] = Field(default="Male")

    @field_validator('email')
    @classmethod
    def validate_email(cls, value):
        allowed_domains = ['sbi.com', 'equitas.com']
        domain = value.split('@')[-1]
        if domain not in allowed_domains:
            raise ValueError(f"Email domain must be one of {allowed_domains}")
        return value
    
    @field_validator('name', mode='after')
    @classmethod
    def transform_name(cls, value):
        return value.strip().upper()
    
    @field_validator('age')
    @classmethod
    def validate_age(cls, value):
        if value < 18:
            raise ValueError("Patient must be at least 18 years old")
        return value
    
    @model_validator(mode='after')
    def validate_emergency_contact(cls, model):
        if model.age > 60 and 'emergency_contact' not in model.contact_info:
            raise ValueError("Patients over 60 must have an emergency contact")
        return model
    
    @computed_field
    @property
    def bmi(self) -> float:
        return round(self.weight / ((self.height / 100) ** 2), 2)

Gender will come as male by default

In [16]:
patient_info = {'name': 'benky', 'email': 'benky@sbi.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '65', 'weight': 70.5, 'height': 175.0, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890', 'emergency_contact': '9876543210'}, 'address': address1}
patient2 = Patient(**patient_info)
print(patient2)

name='BENKY' email='benky@sbi.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=65 weight=70.5 height=175.0 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890', 'emergency_contact': '9876543210'} address=Address(city='New York', state='NY', pin='10001') gender='Male' bmi=23.02


Note that we haven't defined the gender at all inside the patient_info, but as the default value is Male we get it, but what if I want to restrict it?

In [17]:
temp = patient1.model_dump(exclude_unset=True)
print(temp)

{'name': 'BENKY', 'email': 'benky@sbi.com', 'linkedin_url': AnyUrl('https://linkedin.com/in/benky'), 'age': 65, 'weight': 70.5, 'height': 175.0, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890', 'emergency_contact': '9876543210'}, 'address': {'city': 'New York', 'state': 'NY', 'pin': '10001'}, 'bmi': 23.02}
